# QLoRA Fine-tune — Text-to-SQL on Spider (Colab T4)

Phase 1 of the Multi-Agent Analyst System. Fine-tunes a 4-bit base model with QLoRA so it reliably turns *(schema + question)* into SQL. The exported adapter is loaded later by `src/tools/text_to_sql.py` as a live agent tool.

**Defaults (per spec):** 4-bit base, LoRA `r=16`, `alpha=32`, target *all* linear layers, LR `2e-4`, gradient checkpointing. Fits on a **free Colab T4**.

**Runtime:** set `Runtime > Change runtime type > T4 GPU` before running.

In [ ]:
# 1. Install — LOCAL GPU (RTX A4500). Assumes a working CUDA + PyTorch already.
%%capture
!pip install unsloth
!pip install --upgrade datasets trl peft accelerate bitsandbytes
# On Colab instead use:
# !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" datasets

In [ ]:
# 2. Config — tuned for a single NVIDIA RTX A4500 (20 GB, Ampere / bf16)
BASE_MODEL    = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"  # 4-bit base
MAX_SEQ_LEN   = 1024          # schema+question+SQL is short; 1024 is plenty & much faster than 2048
LORA_R        = 16
LORA_ALPHA    = 32
LORA_DROPOUT  = 0.0
LEARNING_RATE = 2e-4
EPOCHS        = 1
BATCH_SIZE    = 8             # 20 GB handles 8B-4bit @ seq 1024 with packing (drop to 4 if OOM)
GRAD_ACCUM    = 2             # effective batch = 16
MAX_STEPS     = 600           # TIME CAP — enough to show a clear before/after delta (~10-15 min).
                              # Set to 0 to train a full epoch instead (slower, marginally better).
MAX_TRAIN     = 0             # 0 = all rows; MAX_STEPS already bounds wall-clock
OUTPUT_DIR    = "outputs"
ADAPTER_DIR   = "adapter"
HF_REPO       = ""            # e.g. "your-username/llama31-sql-qlora" to push to the Hub

## Data — build Spider instruction dataset

Either upload `finetune/data/train.jsonl` (produced by `prepare_spider.py`) to the Colab session, or run the prep inline below. The prompt template **must match** `prepare_spider.py` and `text_to_sql.py` exactly.

In [ ]:
# 3. Build the dataset (schema-inline, Spider-derived — loads cleanly, no script)
from datasets import load_dataset

SYSTEM = (
    "You are a precise text-to-SQL engine. Given a database schema and a question, "
    "output a single valid SQLite query that answers it. Output ONLY the SQL, no prose."
)
PROMPT_TEMPLATE = (
    "{system}\n\n### Database schema:\n{schema}\n\n### Question:\n{question}\n\n### SQL:\n"
)

# Each row: question, context (CREATE TABLE schema), answer (SQL). Spider + WikiSQL
# derived; schema is inline so no fragile tables.json assembly. (The canonical
# `spider` loader is script-based and breaks on newer `datasets`.)
ds = load_dataset("b-mc2/sql-create-context", split="train")

EOS = "<|eot_id|>"  # Llama-3.1 end token
def to_text(ex):
    prompt = PROMPT_TEMPLATE.format(system=SYSTEM, schema=ex["context"], question=ex["question"])
    return {"text": prompt + ex["answer"].strip() + EOS}

train_ds = ds.map(to_text, remove_columns=ds.column_names)
if MAX_TRAIN:
    train_ds = train_ds.select(range(min(MAX_TRAIN, len(train_ds))))
print(train_ds)
print(train_ds[0]["text"][:600])

In [ ]:
# 4. Load 4-bit base + attach LoRA (all linear layers)
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],  # all linear
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

In [ ]:
# 5. Train (QLoRA via TRL SFTTrainer) — optimized for RTX A4500
# Speedups vs the T4 defaults: bf16 (Ampere), packing=True (no padding waste),
# bigger batch, shorter seq len, and a max_steps time cap.
# IMPORTANT: pass tokenizer=tokenizer directly — do NOT rename it to processing_class.
from trl import SFTTrainer
from unsloth import is_bfloat16_supported

targs = dict(
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    warmup_steps=10,
    num_train_epochs=EPOCHS,
    max_steps=MAX_STEPS if MAX_STEPS else -1,   # -1 => train by epochs
    learning_rate=LEARNING_RATE,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),               # A4500 -> bf16 = True
    logging_steps=20,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
    output_dir=OUTPUT_DIR,
    report_to="none",
)

# packing=True concatenates short examples to fill MAX_SEQ_LEN -> far fewer steps,
# big throughput win for short SQL samples. dataset_num_proc speeds up tokenization.
try:
    from trl import SFTConfig
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_ds,
        args=SFTConfig(dataset_text_field="text", max_seq_length=MAX_SEQ_LEN,
                       packing=True, dataset_num_proc=2, **targs),
    )
except (ImportError, TypeError):
    from transformers import TrainingArguments
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_ds,
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LEN,
        packing=True,
        dataset_num_proc=2,
        args=TrainingArguments(**targs),
    )

trainer.train()

In [ ]:
# 6. Save the LoRA adapter (small — just the adapter, not the base)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Saved adapter to", ADAPTER_DIR)

# Optional: push to HF Hub so the app / Spaces can pull it at runtime.
# Locally, either set HF_TOKEN in the environment or run `huggingface-cli login` first.
if HF_REPO:
    import os
    from huggingface_hub import login
    token = os.environ.get("HF_TOKEN")
    login(token) if token else login()   # falls back to cached credentials
    model.push_to_hub(HF_REPO)
    tokenizer.push_to_hub(HF_REPO)
    print("Pushed to", HF_REPO)

# Then point the app at it: set SQL_ADAPTER_REPO=<HF_REPO> in .env, OR copy the
# adapter/ folder into the repo at finetune/adapter/.

In [ ]:
# 7. Quick sanity inference
FastLanguageModel.for_inference(model)
schema = "CREATE TABLE singer (name TEXT, country TEXT, age INT);"
q = "What are the names of singers from France, oldest first?"
prompt = PROMPT_TEMPLATE.format(system=SYSTEM, schema=schema, question=q)
ids = tokenizer(prompt, return_tensors="pt").to("cuda")
out = model.generate(**ids, max_new_tokens=128, do_sample=False)
print(tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True))

## Done — what to copy off the GPU box

After all cells run you have, on this machine:
1. **`eval_report.json`** + the printed `BEFORE → AFTER` execution-accuracy numbers — paste them into the repo's `README.md` proof-point table and `finetune/README.md`.
2. **The adapter on HF Hub** (if you set `HF_REPO` and were logged in) — note the repo id, e.g. `your-username/llama31-sql-qlora`.

Then on your normal machine (no GPU needed): set `SQL_ADAPTER_REPO=<that repo id>` in `.env`, update the README numbers, and commit. (The adapter only *runs* on a GPU — locally the app keeps using the Groq fallback, which is expected and documented.)

In [ ]:
# 8. Download Spider dev set + real databases (for execution accuracy)
import glob, json, os
from huggingface_hub import snapshot_download

# xlangai/spider is the canonical Spider repo on the Hub and ships the sqlite DBs.
spider_dir = snapshot_download("xlangai/spider", repo_type="dataset")
_find = lambda p: (glob.glob(os.path.join(spider_dir, "**", p), recursive=True) or [None])[0]
dev_json, tables_json = _find("dev.json"), _find("tables.json")
_dbs = glob.glob(os.path.join(spider_dir, "**", "database"), recursive=True)
DB_DIR = _dbs[0] if _dbs else None
print("dev.json:", dev_json, "\ntables.json:", tables_json, "\ndatabase/:", DB_DIR)
assert dev_json and tables_json and DB_DIR, (
    "Spider DBs not found in the snapshot. Fallback: download Spider 1.0 from "
    "https://yale-lily.github.io/spider , unzip, and set DB_DIR=<...>/database, "
    "dev_json=<...>/dev.json, tables_json=<...>/tables.json manually."
)

dev = json.load(open(dev_json))
tables = {t["db_id"]: t for t in json.load(open(tables_json))}

def schema_for(db_id):
    m = tables[db_id]
    names, cols, types = m["table_names_original"], m["column_names_original"], m["column_types"]
    per = {i: [] for i in range(len(names))}
    for ci, (ti, cn) in enumerate(cols):
        if ti != -1:
            per[ti].append(f"{cn} {types[ci].upper()}")
    return "\n".join(f"CREATE TABLE {names[ti]} ({', '.join(per[ti])});" for ti in range(len(names)))

print("dev examples:", len(dev))

In [ ]:
# 9. Execution accuracy: BASE vs FINE-TUNED on real Spider dev DBs (runs on this GPU)
import json, os, sqlite3, torch
from unsloth import FastLanguageModel

LIMIT = 200   # held-out dev questions to score (raise for a tighter estimate)

# A separate frozen base model for the "before" number (unambiguous vs the adapter).
# Two 4-bit 8B models fit in 20 GB.
base_model_eval, base_tok = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=MAX_SEQ_LEN, load_in_4bit=True)
FastLanguageModel.for_inference(base_model_eval)
FastLanguageModel.for_inference(model)

def _clean(t):
    t = t.replace("```sql", "").replace("```", "").strip()
    for s in ("\n###", "\nQuestion:", "\n--", "\n\n"):
        if s in t:
            t = t.split(s)[0]
    t = t.strip()
    return (t.split(";")[0] + ";").strip() if ";" in t else t

def _gen(m, tok, prompt):
    ids = tok(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = m.generate(**ids, max_new_tokens=128, do_sample=False,
                         pad_token_id=tok.eos_token_id)
    return _clean(tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True))

def _run(db, sql):
    try:
        con = sqlite3.connect(f"file:{db}?mode=ro", uri=True)
        con.text_factory = lambda b: b.decode("utf-8", "ignore")
        rows = con.execute(sql).fetchall(); con.close()
        return True, sorted(map(repr, rows))
    except Exception:
        return False, None

def evaluate(m, tok):
    correct = runnable = total = 0
    for ex in dev[:LIMIT]:
        db = os.path.join(DB_DIR, ex["db_id"], ex["db_id"] + ".sqlite")
        if not os.path.exists(db):
            continue
        total += 1
        prompt = PROMPT_TEMPLATE.format(system=SYSTEM, schema=schema_for(ex["db_id"]),
                                        question=ex["question"])
        gok, g = _run(db, ex["query"])
        pok, p = _run(db, _gen(m, tok, prompt))
        runnable += int(pok)
        correct += int(gok and p == g)
    return dict(total=total,
                exec_accuracy=round(correct / total, 4) if total else 0.0,
                runnable_rate=round(runnable / total, 4) if total else 0.0)

before = evaluate(base_model_eval, base_tok); print("BEFORE (base):     ", before)
after  = evaluate(model, tokenizer);          print("AFTER  (fine-tuned):", after)

report = dict(base_model=BASE_MODEL, n=before["total"], before=before, after=after,
              exec_accuracy_delta=round(after["exec_accuracy"] - before["exec_accuracy"], 4))
open("eval_report.json", "w").write(json.dumps(report, indent=2))
print(f"\nEXEC-ACCURACY  {before['exec_accuracy']:.1%} -> {after['exec_accuracy']:.1%}  "
      f"(delta {report['exec_accuracy_delta']:+.1%})")
print("Saved eval_report.json. Copy these numbers into README.md and finetune/README.md.")

## Next: measure execution accuracy (before vs after)

Download Spider's `database/` folder, then run `finetune/evaluate_sql.py` to get the **before/after exec-accuracy** numbers for the README proof-point table.

```bash
python finetune/evaluate_sql.py --spider-db-dir spider/database --adapter adapter --limit 200
```